## Import Region data into MongoDB
The goal is to import the following location (region) data into our database:

- Countries and Continents (with geometry and coordinates)
- States and Districts (with geometry and coordinates, postal code, address region)
- Cities, localities and neighbourhoods (with geometry, postal code, address region, address locality, street address)

Having this data locally makes it cheaper and reduces latency for us.

#### Install Dependencies

In [2]:
!uv sync --python 3.13

zsh:1: command not found: uv


#### Import Libraries

In [38]:
import geopandas as gpd
import os
import httpx
import zipfile
import tenacity

### Data Extraction Helpers

Let's define helper functions to help us download and cache datasets for processing.

In [54]:

DATA_DIR = "data"

@tenacity.retry(
    stop=tenacity.stop_after_attempt(5),            # retry up to 5 times
    wait=tenacity.wait_exponential(multiplier=1, min=2, max=20),  # exponential backoff
    reraise=True
)
async def fetch_file(url: str, filepath: str):
    """Fetch a file asynchronously with retries."""
    timeout = httpx.Timeout(30.0, connect=10.0)
    async with httpx.AsyncClient(timeout=timeout) as client:
        response = await client.get(url)
        response.raise_for_status()
        with open(filepath, "wb") as f:
            f.write(response.content)
    return filepath


async def download_and_extract(url: str, extract_to: str = DATA_DIR, force: bool = False) -> list[str]:
    """
    Async download and extract a file (zip or direct) from a URL.

    Retries handled by tenacity.
    """
    os.makedirs(extract_to, exist_ok=True)
    filename = os.path.basename(url)
    filepath = os.path.join(extract_to, filename)

    if not os.path.exists(filepath) or force:
        print(f"Downloading from {url}...")
        await fetch_file(url, filepath)
        print("Download complete.")
    else:
        print(f"File already exists: {filepath}, skipping download.")

    extracted_files = []
    if zipfile.is_zipfile(filepath):
        with zipfile.ZipFile(filepath, "r") as z:
            z.extractall(extract_to)
            extracted_files = [os.path.join(extract_to, name) for name in z.namelist()]
        print("Extraction complete.")
    else:
        extracted_files = [filepath]

    return extracted_files

### Extract Continent Data

Let's start from the top- extract continent data first

In [55]:
def build_aliases(world: gpd.GeoDataFrame):
    """Build a flat list of alternative names for each continent."""

    continent_altnames = {}
    for continent, group in world.groupby("CONTINENT"):
        continent_altnames[continent] = []
    return continent_altnames


async def load_continents():
    """Load continent geometries with name, centroid coordinates, and alternative names."""
    extracted_files = await download_and_extract(
        "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
    )

    shapefile = next((f for f in extracted_files if f.endswith(".shp")), None)
    if shapefile is None:
        raise FileNotFoundError("No shapefile found in extracted dataset.")

    # Load countries
    world = gpd.read_file(shapefile)

    # Dissolve into continents
    continents = world.dissolve(by="CONTINENT").reset_index()[["CONTINENT", "geometry"]]

    # Reproject to a projected CRS before centroid calculation
    continents_proj = continents.to_crs("ESRI:54034")  # Equal Earth projection
    centroids_proj = continents_proj.centroid

    # Convert centroids back to lat/lon (WGS84)
    centroids_wgs84 = gpd.GeoSeries(centroids_proj, crs="ESRI:54034").to_crs("EPSG:4326")

    # Assign coordinates as [lat, lon]
    continents["coordinates"] = centroids_wgs84.apply(lambda p: [p.y, p.x])

    # Build alt names
    continent_altnames = build_aliases(world)
    continents["aliases"] = continents["CONTINENT"].map(continent_altnames)

    # Rename column for clarity
    continents = continents.rename(columns={"CONTINENT": "name"})

    return continents

continents = await load_continents()
continents

File already exists: data/ne_110m_admin_0_countries.zip, skipping download.
Extraction complete.


,name,geometry,coordinates,aliases
0,Africa,"MULTIPOLYGON (((-11.43878 6.78592, -11.70819 6...","[6.032181157557932, 18.40040406024284]",[]
1,Antarctica,"MULTIPOLYGON (((-61.13898 -79.98137, -60.61012...","[-75.50522868903988, 35.64346078762372]",[]
2,Asia,"MULTIPOLYGON (((48.67923 14.0032, 48.23895 13....","[29.888226373617375, 84.71223389814556]",[]
3,Europe,"MULTIPOLYGON (((-53.55484 2.3349, -53.77852 2....","[56.65066091381333, 73.8348995832024]",[]
4,North America,"MULTIPOLYGON (((-155.22217 19.23972, -155.5421...","[47.14289012790487, -96.36461725674786]",[]
5,Oceania,"MULTIPOLYGON (((147.91405 -43.21152, 147.56456...","[-24.573805155566546, 136.50696195370307]",[]
6,Seven seas (open ocean),"POLYGON ((68.935 -48.625, 69.58 -48.94, 70.525...","[-49.30442568487861, 69.53123972063625]",[]
7,South America,"MULTIPOLYGON (((-68.63999 -55.58002, -69.2321 ...","[-13.587586113083029, -60.64773056426989]",[]


In [51]:
# Drop Seven Seas
# note: Oceania is taken as a continent, and not Australia here- which is referred to as a country.
continents = continents[continents["name"] != "Seven seas (open ocean)"]
continents = continents.reset_index(drop=True)
continents

,name,geometry,coordinates,aliases
0,Africa,"MULTIPOLYGON (((-11.43878 6.78592, -11.70819 6...","[6.032181157557932, 18.40040406024284]",[]
1,Antarctica,"MULTIPOLYGON (((-61.13898 -79.98137, -60.61012...","[-75.50522868903988, 35.64346078762372]",[]
2,Asia,"MULTIPOLYGON (((48.67923 14.0032, 48.23895 13....","[29.888226373617375, 84.71223389814556]",[]
3,Europe,"MULTIPOLYGON (((-53.55484 2.3349, -53.77852 2....","[56.65066091381333, 73.8348995832024]",[]
4,North America,"MULTIPOLYGON (((-155.22217 19.23972, -155.5421...","[47.14289012790487, -96.36461725674786]",[]
5,Oceania,"MULTIPOLYGON (((147.91405 -43.21152, 147.56456...","[-24.573805155566546, 136.50696195370307]",[]
6,South America,"MULTIPOLYGON (((-68.63999 -55.58002, -69.2321 ...","[-13.587586113083029, -60.64773056426989]",[]


### Extract Country Data
Next, let's import all the countries in the continents we have extracted so far.

In [61]:
def build_aliases(countries: gpd.GeoDataFrame):
    """Build a flat list of alternative names for each country."""
    alt_name_columns = [
        col for col in countries.columns
        if col.startswith("NAME_") or col in ["NAME_ALT", "NAME_LONG", "FORMAL_EN", "FORMAL_FR"]
    ]

    country_altnames = {}
    for _, row in countries.iterrows():
        names = set()
        for col in alt_name_columns:
            v = row[col]
            if v not in [None, "", -99] and isinstance(v, str):
                names.add(str(v))  # ensure all names are strings
        country_altnames[row["ADMIN"]] = sorted(names)

    return country_altnames


async def load_countries():
    """Load all countries with name, centroid coordinates, and alternative names."""
    extracted_files = await download_and_extract(
        "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
    )

    shapefile = next((f for f in extracted_files if f.endswith(".shp")), None)
    if shapefile is None:
        raise FileNotFoundError("No shapefile found in extracted dataset.")

    # Load countries
    countries = gpd.read_file(shapefile)

    # Build aliases before dropping columns
    country_altnames = build_aliases(countries)

    # Keep only useful columns
    countries = countries[['ADMIN', 'ISO_A2', 'geometry']]

    # Project to a suitable projected CRS for accurate centroid computation
    countries_proj = countries.to_crs("ESRI:54034")  # Equal Earth
    centroids_proj = countries_proj.centroid
    centroids_wgs84 = gpd.GeoSeries(centroids_proj, crs="ESRI:54034").to_crs("EPSG:4326")

    # Assign coordinates as [lat, lon]
    countries['coordinates'] = centroids_wgs84.apply(lambda p: [p.y, p.x])

    # Map aliases
    countries['aliases'] = countries['ADMIN'].map(country_altnames)

    # Optionally rename columns for clarity
    countries = countries.rename(columns={"ADMIN": "name", "ISO_A2": "iso_a2"})

    return countries

countries = await load_countries()
countries

File already exists: data/ne_110m_admin_0_countries.zip, skipping download.
Extraction complete.


,name,iso_a2,geometry,coordinates,aliases
0,Fiji,FJ,"MULTIPOLYGON (((180 -16.06713, 180 -16.55522, ...","[-17.31277520899394, 163.77640725005324]","[Fidji, Fidschi, Fidzsi-szigetek, Fidżi, Figi,..."
1,United Republic of Tanzania,TZ,"POLYGON ((33.90371 -0.95, 34.07262 -1.05982, 3...","[-6.237730725476361, 34.74766857623911]","[Tansania, Tanzania, Tanzanie, Tanzanya, Tanzá..."
2,Western Sahara,EH,"POLYGON ((-8.66559 27.65643, -8.66512 27.58948...","[24.24928060599045, -12.162183576752351]","[Batı Sahra, Nyugat-Szahara, Sahara Barat, Sah..."
3,Canada,CA,"MULTIPOLYGON (((-122.84 49, -122.97421 49.0025...","[57.62966510888876, -97.64486003045715]","[Canada, Canadá, Kanada, Καναδάς, Канада, קנדה..."
4,United States of America,US,"MULTIPOLYGON (((-122.84 49, -120 49, -117.0312...","[42.075007554015286, -107.5447286816333]","[Amerika Birleşik Devletleri, Amerika Serikat,..."
...,...,...,...,...,...
172,Republic of Serbia,RS,"POLYGON ((18.82982 45.90887, 18.82984 45.90888...","[44.21138805382481, 20.828267375143835]","[Republic of Serbia, Serbia, Serbie, Serbien, ..."
173,Montenegro,ME,"POLYGON ((20.0707 42.58863, 19.80161 42.50009,...","[42.78633660560693, 19.285947508559723]","[Czarnogóra, Karadağ, Montenegro, Montenegró, ..."
174,Kosovo,-99,"POLYGON ((20.59025 41.85541, 20.52295 42.21787...","[42.57742521451801, 20.895365849100475]","[Kosova, Kosovo, Kosowo, Koszovó, Republic of ..."
175,Trinidad and Tobago,TT,"POLYGON ((-61.68 10.76, -61.105 10.89, -60.895...","[10.427976277378917, -61.330418679071556]","[Republic of Trinidad and Tobago, Trinidad and..."


Let's filter out countries with an invalid alpha2 code

In [63]:
countries = countries[countries['iso_a2'] != "-99"]
countries = countries.reset_index(drop=True)
countries


,name,iso_a2,geometry,coordinates,aliases
0,Fiji,FJ,"MULTIPOLYGON (((180 -16.06713, 180 -16.55522, ...","[-17.31277520899394, 163.77640725005324]","[Fidji, Fidschi, Fidzsi-szigetek, Fidżi, Figi,..."
1,United Republic of Tanzania,TZ,"POLYGON ((33.90371 -0.95, 34.07262 -1.05982, 3...","[-6.237730725476361, 34.74766857623911]","[Tansania, Tanzania, Tanzanie, Tanzanya, Tanzá..."
2,Western Sahara,EH,"POLYGON ((-8.66559 27.65643, -8.66512 27.58948...","[24.24928060599045, -12.162183576752351]","[Batı Sahra, Nyugat-Szahara, Sahara Barat, Sah..."
3,Canada,CA,"MULTIPOLYGON (((-122.84 49, -122.97421 49.0025...","[57.62966510888876, -97.64486003045715]","[Canada, Canadá, Kanada, Καναδάς, Канада, קנדה..."
4,United States of America,US,"MULTIPOLYGON (((-122.84 49, -120 49, -117.0312...","[42.075007554015286, -107.5447286816333]","[Amerika Birleşik Devletleri, Amerika Serikat,..."
...,...,...,...,...,...
167,North Macedonia,MK,"POLYGON ((22.38053 42.32026, 22.88137 41.9993,...","[41.60305470453288, 21.697076764970955]","[Bắc Macedonia, Kuzey Makedonya, Macedonia Pół..."
168,Republic of Serbia,RS,"POLYGON ((18.82982 45.90887, 18.82984 45.90888...","[44.21138805382481, 20.828267375143835]","[Republic of Serbia, Serbia, Serbie, Serbien, ..."
169,Montenegro,ME,"POLYGON ((20.0707 42.58863, 19.80161 42.50009,...","[42.78633660560693, 19.285947508559723]","[Czarnogóra, Karadağ, Montenegro, Montenegró, ..."
170,Trinidad and Tobago,TT,"POLYGON ((-61.68 10.76, -61.105 10.89, -60.895...","[10.427976277378917, -61.330418679071556]","[Republic of Trinidad and Tobago, Trinidad and..."


### Extract State Data
Next, let's import all the states in the countries we have extracted so far.

In [74]:
def build_state_aliases(states: gpd.GeoDataFrame):
    """Build flat list of alternative names for each state/province."""
    alt_name_columns = [
        col for col in states.columns
        if col.lower().startswith("name") or col in ["name_alt", "name_local", "name_ascii"]
    ]

    state_altnames = {}
    for _, row in states.iterrows():
        names = set()
        for col in alt_name_columns:
            v = row[col]
            if v not in [None, "", -99] and isinstance(v, str):
                names.add(v)
        state_altnames[row["name"]] = sorted(names)
    return state_altnames


async def load_states():
    """Load all states/provinces worldwide with name, centroid coordinates, and aliases."""
    url = "http://naturalearth.s3.amazonaws.com/10m_cultural/ne_10m_admin_1_states_provinces.zip"
    extracted_files = await download_and_extract(url)

    shapefile = next((f for f in extracted_files if f.endswith(".shp")), None)
    if shapefile is None:
        raise FileNotFoundError("No shapefile found in extracted dataset.")

    states = gpd.read_file(shapefile)
    
    print(states['admin'].unique())

    # Build aliases before dropping columns
    state_altnames = build_state_aliases(states)

    # Keep useful columns
    states = states[['name', 'iso_3166_2', 'admin', 'geometry']]

    # Clean ISO codes
    states['iso_3166_2'] = states['iso_3166_2'].apply(lambda x: x if x != -99 else None)

    # Project for accurate centroid
    states_proj = states.to_crs("ESRI:54034")  # Equal Earth projection
    centroids_proj = states_proj.centroid
    centroids_wgs84 = gpd.GeoSeries(centroids_proj, crs="ESRI:54034").to_crs("EPSG:4326")
    states['coordinates'] = centroids_wgs84.apply(lambda p: [p.y, p.x])

    # Map aliases
    states['aliases'] = states['name'].map(state_altnames)

    # Rename columns for clarity
    states = states.rename(columns={
        "name": "state_name",
        "iso_3166_2": "state_code",
        "admin": "country"
    })

    return states

states = await load_states()
states

Download complete.
Extraction complete.
['Argentina' 'Uruguay' 'Indonesia' 'Malaysia' 'Chile' 'Bolivia' 'Peru'
 'Dhekelia Sovereign Base Area' 'Cyprus' 'India' 'China' 'Israel'
 'Gaza Strip' 'West Bank' 'Lebanon' 'Ethiopia' 'S. Sudan' 'Somalia'
 'Kenya' 'Malawi' 'United Republic of Tanzania' 'Syria' 'Somaliland'
 'France' 'Suriname' 'Guyana' 'South Korea' 'North Korea' 'Morocco'
 'Western Sahara' 'Costa Rica' 'Nicaragua' 'Republic of the Congo'
 'Democratic Republic of the Congo' 'Bhutan' 'Ukraine' 'Belarus' 'Namibia'
 'South Africa' 'Saint Martin' 'Sint Maarten' 'United Arab Emirates'
 'Oman' 'Uzbekistan' 'Kazakhstan' 'Tajikistan' 'Lithuania' 'Brazil'
 'Mongolia' 'Russia' 'Czech Republic' 'Germany' 'Estonia' 'Latvia'
 'Norway' 'Sweden' 'Finland' 'Vietnam' 'Cambodia' 'Luxembourg' 'Belgium'
 'Georgia' 'Macedonia' 'Albania' 'Azerbaijan' 'Kosovo' 'Turkey' 'Spain'
 'Laos' 'Kyrgyzstan' 'Armenia' 'Denmark' 'Libya' 'Tunisia' 'Romania'
 'Hungary' 'Slovakia' 'Poland' 'Ireland' 'United Kingdom' 

,state_name,state_code,country,geometry,coordinates,aliases
0,Entre Ríos,AR-E,Argentina,"POLYGON ((-58.20011 -32.44713, -58.20012 -32.4...","[-32.02547013695879, -59.206689862401774]","[Entre Ríos, Entre Ríos eyaleti, Entre-Rios, Έ..."
1,Paysandú,UY-PA,Uruguay,"POLYGON ((-58.20012 -32.4472, -58.20011 -32.44...","[-32.082363780360275, -57.345107641978004]","[Departemen Paysandú, Paysandu Departmanı, Pay..."
2,Kalimantan Timur,ID-KI,Indonesia,"MULTIPOLYGON (((117.70361 4.16341, 117.70361 4...","[1.352699778087046, 116.38394709926088]","[Borneo Wschodnie, Doğu Kalimantan, East Kalim..."
3,Sabah,MY-12,Malaysia,"MULTIPOLYGON (((117.70361 4.16341, 117.70361 4...","[5.345010895470702, 117.13171580564357]","[North Borneo, Sabah, Sabá, Σαμπάχ, Сабах, סבה..."
4,Arica y Parinacota,CL-AP,Chile,"POLYGON ((-69.51009 -17.50659, -69.50611 -17.5...","[-18.485850925370954, -69.63872074583377]","[Arica e Parinacota, Arica et Parinacota, Aric..."
...,...,...,...,...,...,...
4591,Isla,MT-20,Malta,"POLYGON ((14.51073 35.88896, 14.51537 35.89113...","[35.88612913557184, 14.514443273773145]","[Isla, Isla|Città Invicta, L-Isla, Senglea, Σε..."
4592,Cospicua,MT-06,Malta,"POLYGON ((14.52937 35.88156, 14.527 35.87772, ...","[35.87927143250959, 14.52043248348041]","[Bormla, Cospicua, Κοσπίκουα, Бормла, קוספיקאה..."
4593,Pietà,MT-41,Malta,"POLYGON ((14.49801 35.89251, 14.49616 35.8883,...","[35.88921922041928, 14.492441073423253]","[Pietà, Πιετά, Пьета, Пієта, פיאטה, بييتا, پیئ..."
4594,Msida,MT-34,Malta,"POLYGON ((14.48663 35.88636, 14.48231 35.88979...","[35.894557532084654, 14.48254329099217]","[Msida, Μσίντα, Мсида, Мсіда, מסידה, مسيدا, مس..."
